# 02 — Lake area over time

**In plain words:** notebook 01 told us *which* lakes exist. This notebook measures **how big each lake was over
time**, because a lake that keeps growing is a warning sign.

| Output | What it is |
| --- | --- |
| `lake_area_yearly.csv` | for **all 552 lakes**, the largest water area in each melt season (May–October), **2000–2025**, from Landsat |
| `lake_area_weekly.csv` | for the **3 Tier 2 lakes**, the water area **every week, 2015–2025**, from optical satellites (Sentinel-2 from 2017, Landsat 8/9) |
| `optical_looks.csv` | every single clear satellite look behind the weekly table |
| `radar_test.csv` | why the weekly record does **not** use radar: the test showing radar cannot see the Shisper lake |
| `event_response.csv` | the check: did each lake's weekly area **drop** when a recorded outburst happened? |
| `fig_02_tier2_weekly.png`, `fig_02_tier2_yearly.png`, `run_info.json` | figures, and a record of versions, parameters and counts |

**Steps:** 1 load the lakes from notebook 01 → 2 yearly area from Landsat → 3a radar test → 3 weekly area from optical satellites → 4 checks → 5 save → 6 figures.

**How to run:** open from the GitHub link, Runtime → Run all, approve the pop-ups. Send back the cells marked
**📋 REPORT** and the two figures.

## 0. Setup
Same as notebook 01: use the tested code in `src/karaglof/` from the GitHub repository. The lake list is read from
the repository too (`data/labels/`), so this notebook uses exactly the committed v0.2 lakes.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/inamjadoon/KaraGlof.git"
REPO_DIR = "/content/KaraGlof"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import json
import time
from datetime import UTC, datetime
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import drive

from karaglof import area as ka

ee.Authenticate()
ee.Initialize(project="glof-497313")

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/KaraGlof/02_lake_area")
OUT.mkdir(parents=True, exist_ok=True)
LABELS = Path(REPO_DIR) / "data" / "labels"
EXPORT_FOLDER = "KaraGlof_exports"
print("outputs ->", OUT)

## Parameters
Every choice in one place, so each one can be named and defended.
* **Yearly (Landsat only, all years).** Using the *same* satellite family every year means a lake cannot appear to
  "grow" in 2017 just because a sharper satellite (Sentinel-2) started then. Sentinel-2 is therefore not used for the
  yearly record (and never before 2017, a project rule).
* **Season May–October:** the melt season. It includes the spring filling of ice-dammed lakes such as Shisper, which
  burst in May–June.
* **Water test:** NDWI = (green − near-infrared) / (green + near-infrared) above `NDWI_MIN`, seen in at least
  `MIN_DETECTIONS` clear looks. One look can be fooled by a shadow; two independent looks rarely are.
* **No silent filling:** a lake-year with fewer than `MIN_CLEAR_OBS` clear looks gets an **empty** area.
* **Weekly (optical):** every clear Landsat 8/9 look and, from 2017, every Sentinel-2 look. A look is used only if
  at least `OPT_MIN_CLEAR_FRAC` of the lake zone is cloud-free. Weeks without such a look stay **empty**.
* **Radar (tested, not used):** step 3a shows Sentinel-1 radar cannot see the ice-covered Shisper lake.

In [ ]:
YEARS = list(range(2000, 2026))
SEASON = ("05-01", "11-01")              # May 1 to Oct 31 (end date is exclusive)
LAKE_BUFFER_M = 60                       # look 2 Landsat pixels around each outline, so growth is captured
TIER2_BUFFER_M = 300                     # Tier 2 lakes can grow far beyond their mapped outline
NDWI_MIN = 0.1                           # water threshold for Landsat surface reflectance
MIN_DETECTIONS = 2                       # a pixel counts as water if seen as water in >= 2 clear looks
MIN_CLEAR_OBS = 3                        # fewer clear looks in a season -> area left empty (not filled)

WEEK_START, WEEK_END = "2015-01-01", "2026-01-01"
S2_FROM = "2017-01-01"                   # project rule: no Sentinel-2 before 2017
S2_NDWI_MIN = 0.2                        # Sentinel-2 (top-of-atmosphere) water threshold, as in notebook 01 step 3b
S2_CLOUD_PROB_MAX = 40                   # s2cloudless: a pixel is cloud if cloud probability >= 40%
OPT_MIN_CLEAR_FRAC = 0.8                 # use a look only if >= 80% of the lake zone is clear
EVENT_WINDOW_WEEKS = 4                   # check: mean area 4 weeks before vs 4 weeks after each outburst
S1_SPECKLE_M = 30                        # radar test only: smooth speckle over 30 m
UTM = "EPSG:32643"

## Helper: bring Earth Engine results into Python
The same cached-export helper as notebook 01: big results are exported to Drive (they keep running if Colab
disconnects; watch them at <https://code.earthengine.google.com/tasks>), and a rerun reads the saved file instead of
recomputing it.

In [ ]:
EXPORTS = Path("/content/drive/MyDrive") / EXPORT_FOLDER

def export(fc, name, fmt="CSV"):
    path = EXPORTS / f"{name}.{fmt.lower()}"
    if path.exists():
        print(f"{name}: cached file found, nothing to recompute")
        return path, None
    desc = f"karaglof_{name}"
    for t in ee.batch.Task.list()[:30]:
        st = t.status()
        if st.get("description") == desc and st.get("state") in ("READY", "RUNNING"):
            print(f"{name}: the same export is already running, waiting for it")
            return path, t
    task = ee.batch.Export.table.toDrive(collection=fc, description=desc, folder=EXPORT_FOLDER,
                                         fileNamePrefix=name, fileFormat=fmt)
    task.start()
    print(f"{name}: export started (watch it at https://code.earthengine.google.com/tasks)")
    return path, task

def read_export(path, task):
    t0 = time.time()
    while task is not None and task.active():
        print(f"\r{path.stem}: {task.status()['state']} for {(time.time() - t0) / 60:.0f} min ", end="")
        time.sleep(30)
    if task is not None and task.status()["state"] != "COMPLETED":
        raise RuntimeError(task.status())
    for _ in range(60):
        if path.exists():
            break
        time.sleep(10)
    print(f"\n{path.stem}: ready")
    return pd.read_csv(path)

def to_ee(gdf):
    # lake outlines (lon/lat, rounded to ~0.1 m) as an Earth Engine collection with only the lake_id
    small = gdf[["lake_id", "geometry"]].copy()
    small["geometry"] = small.geometry.set_precision(1e-6)
    gj = json.loads(small.to_json())
    return ee.FeatureCollection([ee.Feature(f) for f in gj["features"]])

## 1. Load the lakes from notebook 01
Reads the committed v0.2 tables. Each outline is widened by `LAKE_BUFFER_M` (Tier 2 lakes by `TIER2_BUFFER_M`), so
water that spreads beyond the mapped outline in a big year is still counted. Outlines are simplified to 5 m (much
finer than a 30 m Landsat pixel) to keep the upload to Earth Engine small.

In [ ]:
lakes = gpd.read_file(LABELS / "lakes_polygons.geojson")
events = pd.read_csv(LABELS / "events.csv")
tier2 = lakes[lakes.tier2].copy()
TIER2_IDS = dict(zip(tier2.name, tier2.lake_id, strict=True))

utm = lakes.to_crs(UTM)
buffer_m = pd.Series(LAKE_BUFFER_M, index=lakes.index).where(~lakes.tier2, TIER2_BUFFER_M)
zones = gpd.GeoDataFrame(lakes[["lake_id", "tier2"]].copy(),
                         geometry=[g.buffer(b).simplify(5) for g, b in zip(utm.geometry, buffer_m, strict=True)],
                         crs=UTM).to_crs("EPSG:4326")
lakes_fc = to_ee(zones)
t2_fc = to_ee(zones[zones.tier2])
box = ee.Geometry.Rectangle(list(zones.total_bounds))
t2_box = ee.Geometry.Rectangle(list(zones[zones.tier2].total_bounds))
print(f"{len(lakes)} lakes ({len(tier2)} Tier 2: {TIER2_IDS}); {len(events)} events")   # 📋 REPORT

## 2. Yearly lake area from Landsat (all lakes, 2000–2025)
For each year, every Landsat 5, 7, 8 and 9 image of the melt season is used. For each image:
* **clear** = not cloud, not cloud shadow (the image's own quality band, `QA_PIXEL`);
* **water** = clear and NDWI > `NDWI_MIN`.

Over the season, a pixel is water if it was water in at least `MIN_DETECTIONS` looks. The lake's yearly area is the
total area of those pixels inside its zone. The number of clear looks over the lake is recorded too, so thin years are
visible and left empty rather than guessed. Landsat 7 has striped gaps after 2003; combining all looks of a season
fills them naturally.

In [ ]:
LANDSAT = [("LANDSAT/LT05/C02/T1_L2", "SR_B2", "SR_B4"),   # Landsat 5: green, near-infrared
           ("LANDSAT/LE07/C02/T1_L2", "SR_B2", "SR_B4"),   # Landsat 7
           ("LANDSAT/LC08/C02/T1_L2", "SR_B3", "SR_B5"),   # Landsat 8
           ("LANDSAT/LC09/C02/T1_L2", "SR_B3", "SR_B5")]   # Landsat 9

def landsat_looks(year):
    start, end = f"{year}-{SEASON[0]}", f"{year}-{SEASON[1]}"
    merged = None
    for cid, green, nir in LANDSAT:
        def look(img, green=green, nir=nir):
            clear = img.select("QA_PIXEL").bitwiseAnd(0b11010).eq(0)    # bits: dilated cloud, cloud, shadow
            sr = img.select([green, nir]).multiply(0.0000275).add(-0.2)  # Collection 2 scale factors
            water = sr.normalizedDifference([green, nir]).gt(NDWI_MIN).And(clear)
            return water.rename("water").addBands(clear.rename("clear")).toUint8()
        col = ee.ImageCollection(cid).filterBounds(box).filterDate(start, end).map(look)
        merged = col if merged is None else merged.merge(col)
    return merged

pixel_km2 = ee.Image.pixelArea().divide(1e6)
images_per_year, yearly_fcs = {}, []
for year in YEARS:
    looks = landsat_looks(year)
    images_per_year[year] = looks.size().getInfo()
    if images_per_year[year] == 0:
        continue
    counts = looks.sum()
    water_km2 = counts.select("water").gte(MIN_DETECTIONS).multiply(pixel_km2).rename("water_km2")
    fc = water_km2.reduceRegions(lakes_fc, ee.Reducer.sum(), 30)
    fc = counts.select("clear").reduceRegions(fc, ee.Reducer.median(), 30)
    yearly_fcs.append(fc.map(lambda f, year=year: ee.Feature(None, {
        "lake_id": f.get("lake_id"), "year": year, "water_km2": f.get("sum"), "clear_obs": f.get("median")})))
print("Landsat images per melt season:", images_per_year)   # 📋 REPORT

TAG = f"ndwi{NDWI_MIN}_det{MIN_DETECTIONS}_buf{LAKE_BUFFER_M}-{TIER2_BUFFER_M}"
raw_yearly = read_export(*export(ee.FeatureCollection(yearly_fcs).flatten(), f"lake_area_yearly_raw_{TAG}"))
yearly = ka.yearly_table(raw_yearly, lakes.lake_id, YEARS, MIN_CLEAR_OBS)
cov = yearly.groupby("year").area_km2.apply(lambda s: s.notna().mean()).round(2)
print(f"lake-years with an area: {yearly.area_km2.notna().mean():.1%}")   # 📋 REPORT
print("share of lakes with an area, per year:", cov.to_dict())

## 3a. Why the weekly record does not use radar 📋 REPORT
The plan was Sentinel-1 radar, because it sees through cloud and calm water looks dark on it. The first run showed
almost no water at Shisper, so this cell tests radar directly: brightness inside each lake outline in the 3 weeks
**before** (lake full) and **after** (lake empty) every outburst **before 2022**. The 2022 Shisper outburst is the
final replay, so it is deliberately left out of any test or calibration.

Result of the first run: at Shisper the full lake (about −9.4 dB) is nearly as bright as the empty valley (−7.9 dB),
and almost none of it is darker than −15 dB. Ice and icebergs on the lake and the steep valley walls make the
radar return bright. So the weekly record uses optical satellites instead (step 3). This is a reportable finding.

In [ ]:
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD").filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV")).select("VV"))
outlines = to_ee(lakes[lakes.tier2])
CAL = {"Shisper": ["2019-06-22", "2020-05-29"],
       "Khurdopin": ["2017-07-28", "2018-05-10", "2019-05-10"]}          # pre-2022 outbursts only
rows = []
for name, dates in CAL.items():
    geom = outlines.filter(ee.Filter.eq("lake_id", TIER2_IDS[name])).geometry()
    for d in dates:
        D = pd.Timestamp(d)
        for window, a, b in [("lake full (before)", D - pd.Timedelta(days=21), D),
                             ("lake empty (after)", D + pd.Timedelta(days=1), D + pd.Timedelta(days=22))]:
            col = s1.filterBounds(geom).filterDate(a.strftime("%Y-%m-%d"), b.strftime("%Y-%m-%d"))
            def stats(img, geom=geom):
                vv = img.focalMedian(S1_SPECKLE_M, "circle", "meters")
                bands = ee.Image.cat(vv.rename("mean_dB"), vv.lt(-18).rename("dark18"), vv.lt(-15).rename("dark15"))
                return ee.Feature(None, bands.reduceRegion(ee.Reducer.mean(), geom, 10)).set(
                    {"date": img.date().format("YYYY-MM-dd"), "orbit": img.get("orbitProperties_pass")})
            for f in col.map(stats).getInfo()["features"]:
                rows.append({"lake": name, "outburst": d, "window": window, **f["properties"]})
radar_test = pd.DataFrame(rows).round(2)
print(radar_test.groupby(["lake", "window"])[["mean_dB", "dark18", "dark15"]].mean().round(2).to_string())

## 3. Weekly lake area from optical satellites (Tier 2 lakes, 2015–2025)
Every clear look at each Tier 2 lake zone:
* **Landsat 8 and 9** (all years): cloud and shadow masked with the image's own `QA_PIXEL` band; water = NDWI > `NDWI_MIN`
  (the same rule as the yearly record);
* **Sentinel-2** from 2017: clouds masked with the s2cloudless cloud-probability layer; water = NDWI > `S2_NDWI_MIN`
  (the same rule that mapped the Shisper outline in notebook 01).

For each look, the notebook records the water area and the share of the zone that was clear (`valid_frac`).
`karaglof.area.weekly_series` keeps looks with at least `OPT_MIN_CLEAR_FRAC` clear and takes the **median per week**.
**Weeks without a clear look stay empty**, never guessed; the number of usable looks per week is kept (`n_obs`).
Known limitations: an ice-covered lake does not count as water, so winter and early-spring areas are low, and cloud
shadow that s2cloudless misses can occasionally look like water.

In [ ]:
zone_km2 = dict(zip(zones.lake_id, zones.to_crs(UTM).area / 1e6, strict=True))

def per_look(img, water, clear, sensor, scale):
    bands = ee.Image.cat(water.multiply(pixel_km2).rename("water_km2"), clear.multiply(pixel_km2).rename("clear_km2"))
    return bands.reduceRegions(t2_fc, ee.Reducer.sum(), scale).map(lambda f: f.set({
        "date": img.date().format("YYYY-MM-dd"), "sensor": sensor}))

def landsat_look(img):
    clear = img.select("QA_PIXEL").bitwiseAnd(0b11010).eq(0)
    sr = img.select(["SR_B3", "SR_B5"]).multiply(0.0000275).add(-0.2)
    water = sr.normalizedDifference(["SR_B3", "SR_B5"]).gt(NDWI_MIN).And(clear)
    return per_look(img, water, clear, "Landsat", 30)

landsat = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
             .filterBounds(t2_box).filterDate(WEEK_START, WEEK_END))

s2 = ee.ImageCollection("COPERNICUS/S2_HARMONIZED").filterBounds(t2_box).filterDate(S2_FROM, WEEK_END)
s2_prob = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(t2_box).filterDate(S2_FROM, WEEK_END)
s2 = ee.ImageCollection(ee.Join.saveFirst("cloud").apply(
    s2, s2_prob, ee.Filter.equals(leftField="system:index", rightField="system:index")))

def s2_look(img):
    img = ee.Image(img)
    clear = ee.Image(img.get("cloud")).select("probability").lt(S2_CLOUD_PROB_MAX)
    water = img.normalizedDifference(["B3", "B8"]).gt(S2_NDWI_MIN).And(clear)
    return per_look(img, water, clear, "Sentinel-2", 10)

looks_fc = landsat.map(landsat_look).flatten().merge(s2.map(s2_look).flatten())
OPT_TAG = f"ls{NDWI_MIN}_s2{S2_NDWI_MIN}_cp{S2_CLOUD_PROB_MAX}_buf{TIER2_BUFFER_M}"
obs = read_export(*export(looks_fc.select(["lake_id", "date", "sensor", "water_km2", "clear_km2"]),
                          f"optical_tier2_{OPT_TAG}"))
obs["valid_frac"] = obs.clear_km2 / obs.lake_id.map(zone_km2)
weekly = ka.weekly_series(obs, TIER2_IDS.values(), WEEK_START, "2025-12-31", OPT_MIN_CLEAR_FRAC)
usable = obs[obs.valid_frac >= OPT_MIN_CLEAR_FRAC]
print(f"{len(obs)} looks; {len(usable)} at least {OPT_MIN_CLEAR_FRAC:.0%} clear "
      f"({usable.sensor.value_counts().to_dict()})")   # 📋 REPORT
print("weeks with an area, per lake:",
      weekly.groupby("lake_id").area_km2.apply(lambda s: int(s.notna().sum())).to_dict(), "of", weekly.week_start.nunique())

## 4. Checks 📋 REPORT
1. **Does the radar see the outbursts?** For every outburst since 2015 with an exact day, compare the mean weekly
   area in the 4 weeks before with the 4 weeks after. A real drainage should show a clear **drop** (negative
   `change_pct`). This checks the *data*; no threshold is tuned on it.
2. **Does the weekly record agree with independent maps?** The GLOF database mapped each lake on Planet images just
   before outbursts (carried over from notebook 01). The closest clear optical look within 6 days is compared with it.
3. **No silent filling:** every empty lake-year has fewer than `MIN_CLEAR_OBS` clear looks, and nothing is marked
   filled.

In [ ]:
ev = events[(events.date_certainty == "day") & (events.date >= WEEK_START) & events.lake_id.isin(TIER2_IDS.values())]
response = ka.event_response(weekly, ev, EVENT_WINDOW_WEEKS)
response["lake"] = response.lake_id.map({v: k for k, v in TIER2_IDS.items()})
print(response[["lake", "date", "weeks_with_data_before", "weeks_with_data_after",
                "before_km2", "after_km2", "change_pct"]].to_string(index=False))
drops = response.dropna(subset=["change_pct"])
print(f"\noutbursts with a drop of at least 30%: {int((drops.change_pct <= -30).sum())} of {len(drops)} measurable")

ref = pd.DataFrame(json.loads((LABELS / "run_info.json").read_text())["tier2_outline_validation"])
ref = ref[ref.db_area_km2.notna() & (ref.db_image != "nan")].copy()
looks_ok = usable.copy()
looks_ok["date"] = pd.to_datetime(looks_ok.date)
rows = []
for r in ref.itertuples():
    a = looks_ok[looks_ok.lake_id == TIER2_IDS[r.lake]]
    gap = (a.date - pd.Timestamp(r.db_image)).abs()
    near = a[gap <= pd.Timedelta(days=6)]
    best = near.loc[gap[near.index].idxmin()] if len(near) else None
    rows.append({"lake": r.lake, "planet_image": r.db_image, "planet_km2": r.db_area_km2,
                 "optical_date": best.date.date() if best is not None else None,
                 "sensor": best.sensor if best is not None else None,
                 "optical_km2": round(best.water_km2, 3) if best is not None else None})
print("\nOptical look vs independent Planet-mapped area before outbursts:")
print(pd.DataFrame(rows).to_string(index=False))

empty = yearly[yearly.area_km2.isna()]
assert (empty.n_clear_obs < MIN_CLEAR_OBS).all() and not yearly.filled.any(), "a gap was filled silently"
print(f"\nNo silent filling: {len(empty)} empty lake-years, all with < {MIN_CLEAR_OBS} clear looks. PASS")

## 5. Save outputs to Drive

In [ ]:
yearly.to_csv(OUT / "lake_area_yearly.csv", index=False)
weekly.assign(week_start=weekly.week_start.dt.strftime("%Y-%m-%d")).to_csv(OUT / "lake_area_weekly.csv", index=False)
obs.to_csv(OUT / "optical_looks.csv", index=False)
radar_test.to_csv(OUT / "radar_test.csv", index=False)
response.to_csv(OUT / "event_response.csv", index=False)
run_info = {
    "notebook": "02_lake_area", "run_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    "sources": {"yearly": "Landsat 5/7/8/9 Collection 2 Level 2 (LANDSAT/L*0*/C02/T1_L2)",
                "weekly": "Landsat 8/9 C2 L2 and Sentinel-2 L1C from 2017 with s2cloudless (COPERNICUS/S2_HARMONIZED, S2_CLOUD_PROBABILITY)",
                "radar_test": "Sentinel-1 GRD IW VV (COPERNICUS/S1_GRD), pre-2022 outbursts only",
                "lakes": "data/labels (notebook 01, v0.2)"},
    "parameters": {k: v for k, v in globals().items() if k.isupper() and k not in ("OUT", "REPO_DIR", "LABELS")
                   and isinstance(v, int | float | str | list | tuple | dict)},
    "counts": {"landsat_images_per_year": images_per_year, "lake_years": len(yearly),
               "lake_years_with_area": int(yearly.area_km2.notna().sum()),
               "optical_looks": len(obs), "usable_looks": len(usable),
               "weeks_with_area": int(weekly.area_km2.notna().sum())},
    "radar_test_means": radar_test.groupby(["lake", "window"])[["mean_dB", "dark18", "dark15"]].mean().round(2)
                                  .reset_index().to_dict(orient="records"),
    "event_response": response.to_dict(orient="records"),
    "versions": {"earthengine-api": ee.__version__, "geemap": geemap.__version__, "pandas": pd.__version__},
}
(OUT / "run_info.json").write_text(json.dumps(run_info, indent=2, default=str))
print(sorted(p.name for p in OUT.iterdir()))

## 6. Figures 📋 REPORT
**Weekly figure:** each Tier 2 lake's weekly water area 2015–2025 (optical satellites; gaps = no clear look). Red lines are recorded outbursts, orange dots the
independent Planet-mapped areas. Look for: the lake **filling up before** each red line and **dropping right after**.
**Yearly figure:** the same lakes' largest Landsat area each melt season since 2000.

In [ ]:
names = {v: k for k, v in TIER2_IDS.items()}
fig, axes = plt.subplots(len(TIER2_IDS), 1, figsize=(13, 3.2 * len(TIER2_IDS)), sharex=True)
for ax, (name, lid) in zip(axes, TIER2_IDS.items(), strict=True):
    w = weekly[weekly.lake_id == lid]
    ax.plot(w.week_start, w.area_km2, lw=1, marker=".", ms=3, color="#1f77b4", label="weekly area (optical)")
    for d in events[events.lake_id == lid].date:
        if d >= WEEK_START:
            ax.axvline(pd.Timestamp(d), color="red", lw=1, alpha=0.7)
    r = ref[ref.lake == name]
    ax.scatter(pd.to_datetime(r.db_image), r.db_area_km2, color="orange", zorder=3, label="Planet-mapped (database)")
    ax.set_title(f"{name} ({lid}): weekly water area, red = recorded outburst")
    ax.set_ylabel("km²")
    ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "fig_02_tier2_weekly.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(13, 4))
for name, lid in TIER2_IDS.items():
    y = yearly[yearly.lake_id == lid]
    ax.plot(y.year, y.area_km2, marker="o", label=name)
ax.set_title("Tier 2 lakes: largest water area each melt season (Landsat)")
ax.set_xlabel("year")
ax.set_ylabel("km²")
ax.legend()
fig.savefig(OUT / "fig_02_tier2_yearly.png", dpi=150)
plt.show()

## What to send back
1. The output of every cell marked **📋 REPORT** (steps 1–4).
2. Both figures.

As before: if a cell shows an error, stop there and send it, and don't change thresholds yourself. The weekly
water thresholds in particular must be settled **before** notebook 05, and never adjusted after looking at the
Shisper 2022 replay.